需要注意的是：如果字段没有指定Reducer，那默认的Reducer行为就是覆盖。

也就是说：如果多个Node都更新了State中的同一个字段，默认情况下后执行的Node将覆盖前面Node的更新结果。例如：

In [ ]:
from typing import TypedDict
from langgraph.constants import START, END
from langgraph.graph import StateGraph

# =================1. 定义State数据结构，默认Reducer=================
class DefaultReducerState(TypedDict):
    val: str

# =================2. Node: 定义处理逻辑=================
# node1 覆盖State中的val值为node_1
def node_1(state: DefaultReducerState):
    print(f"node_1 is receiving val: {state['val']}")
    return {"val": "node_1"}
# node2 覆盖State中的val值为node_2
def node_2(state: DefaultReducerState):
    print(f"node_2 is receiving val: {state['val']}")
    return {"val": "node_2"}

# ===================3. 创建Graph===================
graph_builder = StateGraph(DefaultReducerState)         # 3.1.注册State，初始化图Builder
graph_builder.add_node("node_1", node_1)                # 3.2.注册Node
graph_builder.add_node("node_2", node_2)
                                                        # 3.3.注册Edge，定义Node执行顺序
graph_builder.add_edge(START, "node_1")                 # START -> greet
graph_builder.add_edge("node_1", "node_2")              # greet -> uppercase
graph_builder.add_edge("node_2", END)                   # uppercase -> END

graph = graph_builder.compile()                         # 3.4.编译成可执行图

# =================4.调用Graph，传入默认State值=================
result = graph.invoke({"val": "default"})
print(result)

node_1 is receiving val: default
node_2 is receiving val: node_1
{'val': 'node_2'}


1.2.2 自定义Reducer
通过前面的例子我们就能看到默认的Reducer效果了。

如果我们不希望覆盖，则需要自定义Reducer，需要用到Annotated来定义：
- Annotated[type, reducer]

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.constants import START, END
from langgraph.graph import StateGraph


# =================1. 定义State数据结构和Reducer=================
class CustomReducerState(TypedDict):
    count: int                       # 这个字段没有定义Reducer，也就是默认会覆盖旧值
    nodes: Annotated[list[str], add] # 这里使用add作为Reducer，也就是会拼接结果

# =================2. Node: 定义处理逻辑=================
def node_1(state: CustomReducerState):
    print(f'node_1 is receiving count: {state['count']}')
    return {'count': 1, 'nodes': ['node_1']} # 直接修改nodes值，不用自己拼接

def node_2(state: CustomReducerState):
    print(f'node_2 is receiving count: {state['count']}')
    return {'count': 2, 'nodes': ['node_2']} # 直接修改nodes值，不用自己拼接

# ===================3. 创建Graph===================
graph_builder = StateGraph(CustomReducerState)      # 3.1.注册State，初始化图Builder
graph_builder.add_node('node_1', node_1)            # 3.2.注册Node
graph_builder.add_node('node_2', node_2)
                                                    # 3.3.注册Edge，定义Node执行顺序
graph_builder.add_edge(START, 'node_1')                 # START -> greet
graph_builder.add_edge('node_1', 'node_2')              # greet -> uppercase
graph_builder.add_edge('node_2', END)                   # uppercase -> END

graph = graph_builder.compile()                     # 3.4.编译成可执行图

# =================4.调用Graph，传入默认State值=================
result = graph.invoke({'count': 0, 'nodes':[]})
print(result)

node_1 is receiving count: 0
node_2 is receiving count: 1
{'count': 3, 'nodes': ['node_1', 'node_2']}


: 